# 02｜GTEx二分类模型训练、Valid锁定和Test评估

本Notebook包含全部训练、预处理、Tissue one-hot、Valid选模、阈值锁定、全量Test预测、穷尽式1:1轮次、TSS分区评估、置信区间和作图代码，不调用外部训练脚本。

默认`FAST_MODE=True`，只运行一个小规模Random Forest以验证链路。正式实验请确认01已完成，设置新的`RUN_TAG`，再将`FAST_MODE=False`后运行全部单元。

In [4]:
# ======================== 用户统一配置（正式运行前只修改本单元） ========================
from pathlib import Path

PROJECT_ROOT = Path('/vepfs-mlp2/xts001/400107/code/AG_classification/GTEx_self')

# 01 Notebook生成的正表、负表和冻结样本；Train/Test保留原始构成，Valid为逐染色体1:1。
POSITIVE_FILE = PROJECT_ROOT / 'data/positive_gtex_pip_gt_0p9_scored_11modal.parquet'
NEGATIVE_FILE = PROJECT_ROOT / 'data/negative_gtex_pip_lt_0p01_plus_control_scored_11modal.parquet'
SELECTED_SAMPLES_FILE = PROJECT_ROOT / 'data/selected_samples.parquet'
SOURCE_CHROMOSOME_BALANCE_FILE = PROJECT_ROOT / 'data/chromosome_balance.csv'

# Train/Test保持01中冻结的原始类别构成；Valid仍严格逐染色体1:1。
TRAIN_NEG_PER_POS = None
VALID_NEG_PER_POS = None
TEST_NEG_PER_POS = None

# 可选：score11、score33、score33_plus_tissue（主输入）。
FEATURE_RECIPE = 'score33_plus_tissue'
MODELS_TO_RUN = ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM', 'DeepMLP']
GPU_DEVICE = 'cuda:0'
RANDOM_SEED = 42
RUN_TAG = 'GTEX_all'
FAST_MODE = False

# FAST_MODE只运行RF；Train/Test按各染色体原始比例近似缩小，Valid保持1:1。
FAST_POSITIVE_PER_CHROMOSOME = 40
FULL_BOOTSTRAP_REPLICATES = 2000
FAST_BOOTSTRAP_REPLICATES = 100
TSS_AUROC_MIN_POSITIVE = 10
TSS_AUROC_MIN_NEGATIVE = 10

RUN_DIR = PROJECT_ROOT / 'results' / RUN_TAG
MODELS_DIR = RUN_DIR / 'models'
print('RUN_DIR =', RUN_DIR)

RUN_DIR = /vepfs-mlp2/xts001/400107/code/AG_classification/GTEx_self/results/GTEX_all


In [5]:
import hashlib
import json
import math
import os
import random
import time
from copy import deepcopy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.utils.class_weight import compute_sample_weight
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

MODALITIES = [
    'ATAC', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP',
    'RNA_SEQ', 'CONTACT_MAPS', 'SPLICE_SITES', 'SPLICE_SITE_USAGE',
    'SPLICE_JUNCTIONS',
]
SPLIT_CHROMOSOMES = {
    'train': ['chr1', 'chr4', 'chr7', 'chr8', 'chr10', 'chr13', 'chr15'],
    'valid': ['chr2', 'chr5', 'chr11', 'chr14', 'chr17', 'chr20', 'chr22', 'chrX'],
    'test': ['chr3', 'chr6', 'chr9', 'chr12', 'chr16', 'chr18', 'chr19', 'chr21'],
}
RATIO_BY_SPLIT = {
    'train': TRAIN_NEG_PER_POS,
    'valid': VALID_NEG_PER_POS,
    'test': TEST_NEG_PER_POS,
}
VALID_RECIPES = {'score11', 'score33', 'score33_plus_tissue'}
ALL_MODELS = {'LogisticRegression', 'RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM', 'DeepMLP'}
FORBIDDEN_EXACT = {
    'label', 'label_reason', 'sample_source', 'sample_group', 'negative_source',
    'PIP', 'Beta', 'SE', 'CS size', 'CS Unique ID', 'variant_key',
    'chromosome', 'position', 'reference', 'alternate', 'rf_split', 'split',
    'sample_id', 'target_gene_id', 'tissue_match_mode', 'gene_match_mode',
    'Variant Selection Type', 'control_context_group', 'loc_offset_bp',
    'gene_tss', 'tss_distance', 'tss_mapping_method',
}

if FEATURE_RECIPE not in VALID_RECIPES:
    raise ValueError(f'FEATURE_RECIPE必须属于{sorted(VALID_RECIPES)}')
if not set(MODELS_TO_RUN).issubset(ALL_MODELS):
    raise ValueError({'unknown_models': sorted(set(MODELS_TO_RUN) - ALL_MODELS)})
if 'RandomForest' not in MODELS_TO_RUN:
    raise ValueError('MODELS_TO_RUN必须保留RandomForest')
if TEST_NEG_PER_POS is not None:
    raise ValueError('Test主表必须保留全部样本，TEST_NEG_PER_POS只能为None')

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# ======================== 加载01冻结的selected_samples并再次审计 ========================
for path in [POSITIVE_FILE, NEGATIVE_FILE, SELECTED_SAMPLES_FILE, SOURCE_CHROMOSOME_BALANCE_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

selected_full = pd.read_parquet(SELECTED_SAMPLES_FILE)
required = {
    'sample_id', 'variant_key', 'chromosome', 'target_tissue_standardized',
    'label', 'rf_split', 'gene_tss', 'tss_distance', 'tss_mapping_method',
    *MODALITIES,
}
missing = sorted(required - set(selected_full.columns))
if missing:
    raise ValueError({'selected_samples_missing_columns': missing})
if selected_full.sample_id.duplicated().any():
    raise ValueError('selected_samples存在重复sample_id')

def stable_take(frame, n, salt):
    if n > len(frame):
        raise ValueError({'requested': n, 'available': len(frame), 'salt': salt})
    order = frame.sample_id.astype(str).map(
        lambda value: hashlib.sha256(
            f'{RANDOM_SEED}|{salt}|{value}'.encode('utf-8')
        ).hexdigest()
    )
    return frame.assign(_hash=order).sort_values(['_hash', 'sample_id']).head(n).drop(columns='_hash')

def compute_balance(frame):
    table = (
        frame.groupby(['rf_split', 'chromosome', 'label'], observed=True)
        .size().unstack(fill_value=0).rename(columns={0: 'negative', 1: 'positive'}).reset_index()
    )
    for column in ['positive', 'negative']:
        if column not in table:
            table[column] = 0
    table['actual_neg_per_pos'] = table.negative / table.positive
    # None表示保持该split的原始正负比例
    table['target_neg_per_pos'] = pd.to_numeric(
        table.rf_split.map(RATIO_BY_SPLIT),
        errors='coerce',
    )

    fixed_ratio = table['target_neg_per_pos'].notna()

    table['balance_mode'] = np.where(
        fixed_ratio,
        'fixed_ratio',
        'keep_original',
    )

    # None对应的split不检查固定比例
    table['ratio_exact'] = True

    # 只有明确配置了数字比例时才比较
    table.loc[fixed_ratio, 'ratio_exact'] = np.isclose(
        table.loc[fixed_ratio, 'actual_neg_per_pos'].astype(float),
        table.loc[fixed_ratio, 'target_neg_per_pos'].astype(float),
    )
    table['total'] = table.positive + table.negative
    return table.sort_values(['rf_split', 'chromosome'], kind='stable')

# FAST_MODE不做全局随机抽样：Train/Test按每条染色体原始比例近似缩小，Valid保持严格1:1。
if FAST_MODE:
    fast_parts = []
    for split, chromosomes in SPLIT_CHROMOSOMES.items():
        ratio = RATIO_BY_SPLIT[split]
        for chromosome in chromosomes:
            block = selected_full.loc[
                selected_full.rf_split.eq(split) & selected_full.chromosome.eq(chromosome)
            ]
            pos = block.loc[block.label.eq(1)]
            neg = block.loc[block.label.eq(0)]
            if ratio is None:
                keep_pos = min(FAST_POSITIVE_PER_CHROMOSOME, len(pos))
                original_neg_per_pos = len(neg) / len(pos)
                keep_neg = min(len(neg), max(1, round(keep_pos * original_neg_per_pos)))
            else:
                keep_pos = min(FAST_POSITIVE_PER_CHROMOSOME, len(pos), len(neg) // ratio)
                keep_neg = keep_pos * ratio
            if keep_pos < 1:
                raise ValueError(f'FAST_MODE下{split}/{chromosome}没有可用样本')
            fast_parts.extend([
                stable_take(pos, keep_pos, f'fast|{split}|{chromosome}|pos'),
                stable_take(neg, keep_neg, f'fast|{split}|{chromosome}|neg'),
            ])
    selected_run = pd.concat(fast_parts, ignore_index=True, sort=False)
else:
    selected_run = selected_full.copy()

run_balance = compute_balance(selected_run)
if not run_balance.ratio_exact.all():
    raise AssertionError(run_balance.loc[~run_balance.ratio_exact].to_dict('records'))
# 只有明确配置Valid比例时才检查
if VALID_NEG_PER_POS is not None:
    valid_balance = run_balance.loc[
        run_balance.rf_split.eq('valid')
    ]

    if not np.isclose(
        valid_balance['actual_neg_per_pos'].astype(float),
        float(VALID_NEG_PER_POS),
    ).all():
        raise AssertionError(
            f'实际Valid未逐染色体达到neg/pos={VALID_NEG_PER_POS}'
        )

selected_run.to_parquet(RUN_DIR / 'selected_samples.parquet', index=False, compression='zstd')
run_balance.to_csv(RUN_DIR / 'chromosome_balance.csv', index=False)
display(run_balance)
print('本次实际进入训练/验证/测试的rows =', len(selected_run))
print('FAST_MODE =', FAST_MODE)

label,rf_split,chromosome,negative,positive,actual_neg_per_pos,target_neg_per_pos,balance_mode,ratio_exact,total
0,test,chr12,2587,1892,1.367336,NaN,keep_original,True,4479
1,test,chr16,2303,2977,0.773598,NaN,keep_original,True,5280
2,test,chr18,1266,694,1.824207,NaN,keep_original,True,1960
3,test,chr19,2626,3691,0.711460,NaN,keep_original,True,6317
4,test,chr21,766,1125,0.680889,NaN,keep_original,True,1891
5,test,chr3,3221,1841,1.749593,NaN,keep_original,True,5062
6,test,chr6,2616,1284,2.037383,NaN,keep_original,True,3900
7,test,chr9,2400,2047,1.172447,NaN,keep_original,True,4447
8,train,chr1,4923,3994,1.232599,NaN,keep_original,True,8917
9,train,chr10,2947,1915,1.538903,NaN,keep_original,True,4862


本次实际进入训练/验证/测试的rows = 92895
FAST_MODE = False


## Train-only FeatureBuilder

三种输入：

- `score11`：11个signed AG score；
- `score33`：11 signed + 11 absolute + 11 missing indicator；
- `score33_plus_tissue`：score33 + Train拟合的GTEx Tissue one-hot。

连续分数的中位数、均值和标准差只在Train上拟合。Missing和Tissue列保持0/1，不做中心化，因此Valid/Test未知Tissue的全部one-hot列严格为0。

In [7]:
class TrainOnlyFeatureBuilder:
    def __init__(self, recipe, modalities):
        self.recipe = recipe
        self.modalities = list(modalities)
        self.fitted = False

    def fit(self, train_frame):
        self.signed_median = {}
        self.absolute_median = {}
        self.signed_mean = {}
        self.signed_scale = {}
        self.absolute_mean = {}
        self.absolute_scale = {}

        for modality in self.modalities:
            values = pd.to_numeric(train_frame[modality], errors='coerce')
            signed_median = float(values.median()) if values.notna().any() else 0.0
            absolute = values.abs()
            absolute_median = float(absolute.median()) if absolute.notna().any() else 0.0
            signed_filled = values.fillna(signed_median).to_numpy(np.float64)
            absolute_filled = absolute.fillna(absolute_median).to_numpy(np.float64)
            self.signed_median[modality] = signed_median
            self.absolute_median[modality] = absolute_median
            self.signed_mean[modality] = float(signed_filled.mean())
            self.signed_scale[modality] = float(signed_filled.std()) or 1.0
            self.absolute_mean[modality] = float(absolute_filled.mean())
            self.absolute_scale[modality] = float(absolute_filled.std()) or 1.0

        if self.recipe == 'score33_plus_tissue':
            # 类别字典只来自Train；排序后冻结列顺序。
            self.tissue_categories = sorted(
                train_frame.target_tissue_standardized.dropna().astype(str).unique()
            )
        else:
            self.tissue_categories = []
        self.fitted = True
        self.feature_names = self._feature_names()
        self._audit_feature_names()
        return self

    def _feature_names(self):
        names = [f'signed__{m}' for m in self.modalities]
        if self.recipe in {'score33', 'score33_plus_tissue'}:
            names += [f'absolute__{m}' for m in self.modalities]
            names += [f'missing__{m}' for m in self.modalities]
        if self.recipe == 'score33_plus_tissue':
            names += [f'tissue__{t}' for t in self.tissue_categories]
        return names

    def _audit_feature_names(self):
        forbidden_hits = []
        for name in self.feature_names:
            lower = name.lower()
            if name in FORBIDDEN_EXACT:
                forbidden_hits.append(name)
            if any(token in lower for token in ['winning_track', 'match_rule', 'sample_source', 'pip', 'chromosome', 'variant_key']):
                forbidden_hits.append(name)
        if forbidden_hits:
            raise ValueError({'forbidden_features': sorted(set(forbidden_hits))})

    def transform(self, frame):
        if not self.fitted:
            raise RuntimeError('FeatureBuilder必须先在Train上fit')
        blocks = []
        for modality in self.modalities:
            values = pd.to_numeric(frame[modality], errors='coerce')
            filled = values.fillna(self.signed_median[modality]).to_numpy(np.float64)
            blocks.append(((filled - self.signed_mean[modality]) / self.signed_scale[modality])[:, None])

        if self.recipe in {'score33', 'score33_plus_tissue'}:
            for modality in self.modalities:
                values = pd.to_numeric(frame[modality], errors='coerce').abs()
                filled = values.fillna(self.absolute_median[modality]).to_numpy(np.float64)
                blocks.append(((filled - self.absolute_mean[modality]) / self.absolute_scale[modality])[:, None])
            for modality in self.modalities:
                missing = pd.to_numeric(frame[modality], errors='coerce').isna().astype(np.float64).to_numpy()
                blocks.append(missing[:, None])

        if self.recipe == 'score33_plus_tissue':
            tissue = frame.target_tissue_standardized.astype(str)
            for category in self.tissue_categories:
                blocks.append(tissue.eq(category).astype(np.float64).to_numpy()[:, None])

        matrix = np.concatenate(blocks, axis=1).astype(np.float32)
        if matrix.shape[1] != len(self.feature_names):
            raise AssertionError((matrix.shape, len(self.feature_names)))
        return matrix

    def schema(self, train_frame, valid_frame):
        valid_tissues = set(valid_frame.target_tissue_standardized.astype(str))
        unseen_valid = sorted(valid_tissues - set(self.tissue_categories)) if self.tissue_categories else []
        return {
            'recipe': self.recipe,
            'fit_split': 'train only',
            'modalities': self.modalities,
            'n_features': len(self.feature_names),
            'feature_names': self.feature_names,
            'continuous_imputation': 'per-column median fitted on Train only',
            'continuous_standardization': 'mean/std fitted on Train only',
            'binary_columns_standardized': False,
            'tissue_categories_fit_on': 'Train only',
            'tissue_categories': self.tissue_categories,
            'unknown_tissue_rule': 'all-zero tissue one-hot; never add columns',
            'unseen_valid_tissues': unseen_valid,
            'forbidden_fields': sorted(FORBIDDEN_EXACT),
            'signed_median': self.signed_median,
            'absolute_median': self.absolute_median,
            'signed_mean': self.signed_mean,
            'signed_scale': self.signed_scale,
            'absolute_mean': self.absolute_mean,
            'absolute_scale': self.absolute_scale,
        }

train_df = selected_run.loc[selected_run.rf_split.eq('train')].copy()
valid_df = selected_run.loc[selected_run.rf_split.eq('valid')].copy()
# 此处故意不构造X_test；Test要等selection lock落盘后才解封。
builder = TrainOnlyFeatureBuilder(FEATURE_RECIPE, MODALITIES).fit(train_df)
X_train = builder.transform(train_df)
X_valid = builder.transform(valid_df)
y_train = train_df.label.to_numpy(np.int64)
y_valid = valid_df.label.to_numpy(np.int64)

feature_schema = builder.schema(train_df, valid_df)
valid_unknown_one_hot_all_zero = True
if FEATURE_RECIPE == 'score33_plus_tissue' and feature_schema['unseen_valid_tissues']:
    unseen_mask = valid_df.target_tissue_standardized.astype(str).isin(
        feature_schema['unseen_valid_tissues']
    ).to_numpy()
    tissue_start = 33
    valid_unknown_one_hot_all_zero = bool(
        np.all(X_valid[unseen_mask, tissue_start:] == 0)
    )
    if not valid_unknown_one_hot_all_zero:
        raise AssertionError('Valid未知Tissue的one-hot并非全0')
feature_schema['unseen_valid_tissue_one_hot_all_zero'] = valid_unknown_one_hot_all_zero
feature_schema_path = RUN_DIR / 'feature_schema.json'
feature_schema_path.write_text(
    json.dumps(feature_schema, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print('X_train:', X_train.shape, 'X_valid:', X_valid.shape)
print('Train tissue categories:', len(builder.tissue_categories))
print('Valid unseen tissues:', feature_schema['unseen_valid_tissues'])

X_train: (32519, 82) X_valid: (27040, 82)
Train tissue categories: 49
Valid unseen tissues: []


In [8]:
# ======================== 模型、阈值和通用评价函数 ========================
def threshold_from_validation(y_true, probability):
    fpr, tpr, thresholds = roc_curve(y_true, probability)
    finite = np.isfinite(thresholds)
    if not finite.any():
        return 0.5
    objective = tpr[finite] - fpr[finite]  # 等价于最大化Youden J / Balanced Accuracy
    return float(thresholds[finite][int(np.argmax(objective))])

def metric_row(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    return {
        'n': int(len(y_true)),
        'positive': int(np.sum(y_true == 1)),
        'negative': int(np.sum(y_true == 0)),
        'AUROC': float(roc_auc_score(y_true, probability)),
        'AUPRC': float(average_precision_score(y_true, probability)),
        'Balanced_Accuracy': float(balanced_accuracy_score(y_true, prediction)),
        'F1': float(f1_score(y_true, prediction)),
        'Sensitivity': float(sensitivity),
        'Specificity': float(specificity),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'threshold': float(threshold),
    }

def sklearn_model(name):
    fast_trees = 80
    full_trees = 500
    n_trees = fast_trees if FAST_MODE else full_trees
    if name == 'LogisticRegression':
        return LogisticRegression(max_iter=3000, solver='lbfgs', random_state=RANDOM_SEED)
    if name == 'RandomForest':
        return RandomForestClassifier(
            n_estimators=n_trees, max_depth=12, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_SEED,
        )
    if name == 'ExtraTrees':
        return ExtraTreesClassifier(
            n_estimators=n_trees, max_depth=None, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_SEED,
        )
    if name == 'XGBoost':
        device = GPU_DEVICE if torch.cuda.is_available() and GPU_DEVICE.startswith('cuda') else 'cpu'
        return XGBClassifier(
            n_estimators=120 if FAST_MODE else 700, max_depth=6,
            learning_rate=0.05, subsample=0.85, colsample_bytree=0.9,
            objective='binary:logistic', eval_metric='auc', tree_method='hist',
            device=device, random_state=RANDOM_SEED, n_jobs=8,
        )
    if name == 'LightGBM':
        # LightGBM的GPU后端依赖OpenCL；当前环境用CPU更稳定，DeepMLP和XGBoost仍使用GPU_DEVICE。
        device_type = 'cpu'
        return LGBMClassifier(
            n_estimators=120 if FAST_MODE else 700, max_depth=-1,
            num_leaves=63, learning_rate=0.05, subsample=0.85,
            colsample_bytree=0.9, objective='binary', device_type=device_type,
            random_state=RANDOM_SEED, n_jobs=8, verbosity=-1,
        )
    raise ValueError(name)

class DeepMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

def torch_probability(model, matrix, device, batch_size=4096):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(matrix), batch_size):
            x = torch.from_numpy(matrix[start:start + batch_size]).to(device)
            outputs.append(torch.sigmoid(model(x)).cpu().numpy())
    return np.concatenate(outputs)

def fit_deep_mlp(X_tr, y_tr, X_va, y_va):
    device = torch.device(GPU_DEVICE if torch.cuda.is_available() and GPU_DEVICE.startswith('cuda') else 'cpu')
    model = DeepMLP(X_tr.shape[1]).to(device)
    dataset = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr.astype(np.float32)))
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    loader = DataLoader(
        dataset, batch_size=256 if FAST_MODE else 1024, shuffle=True,
        generator=generator, num_workers=0, pin_memory=device.type == 'cuda',
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    max_epochs = 3 if FAST_MODE else 80
    patience = 2 if FAST_MODE else 12
    best_auc, best_state, stale = -np.inf, None, 0
    for epoch in range(max_epochs):
        model.train()
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
        valid_probability = torch_probability(model, X_va, device)
        valid_auc = roc_auc_score(y_va, valid_probability)
        if valid_auc > best_auc + 1e-5:
            best_auc = valid_auc
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    return model, device

def predict_probability(model_name, model_object, matrix):
    if model_name == 'DeepMLP':
        model, device = model_object
        return torch_probability(model, matrix, device)
    return model_object.predict_proba(matrix)[:, 1]

In [9]:
# ======================== 只在Train拟合；只用Valid选模和锁定阈值 ========================
active_models = ['RandomForest'] if FAST_MODE else list(MODELS_TO_RUN)
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
fitted_models = {}
validation_rows = []

for model_name in active_models:
    started = time.time()
    print(f'训练 {model_name} ...')
    if model_name == 'DeepMLP':
        model_object = fit_deep_mlp(X_train, y_train, X_valid, y_valid)
        torch_model, _device = model_object
        torch.save({
            'state_dict': torch_model.state_dict(),
            'n_features': X_train.shape[1],
            'feature_recipe': FEATURE_RECIPE,
            'feature_names': builder.feature_names,
            'random_seed': RANDOM_SEED,
        }, MODELS_DIR / 'DeepMLP.pt')
    else:
        model_object = sklearn_model(model_name)
        model_object.fit(X_train, y_train, sample_weight=sample_weight)
        joblib.dump(model_object, MODELS_DIR / f'{model_name}.joblib')

    valid_probability = predict_probability(model_name, model_object, X_valid)
    threshold = threshold_from_validation(y_valid, valid_probability)
    row = {'model': model_name, **metric_row(y_valid, valid_probability, threshold)}
    row['fit_seconds'] = time.time() - started
    validation_rows.append(row)
    fitted_models[model_name] = model_object
    print(model_name, 'Valid AUROC=', row['AUROC'], 'threshold=', threshold)

validation_metrics = pd.DataFrame(validation_rows).sort_values(
    ['AUROC', 'AUPRC', 'Balanced_Accuracy'], ascending=False, kind='stable'
).reset_index(drop=True)
validation_metrics.to_csv(RUN_DIR / 'validation_metrics.csv', index=False)
display(validation_metrics)

训练 LogisticRegression ...
LogisticRegression Valid AUROC= 0.8842802056344317 threshold= 0.5305042862892151
训练 RandomForest ...
RandomForest Valid AUROC= 0.8521828799543084 threshold= 0.517922489094208
训练 ExtraTrees ...
ExtraTrees Valid AUROC= 0.8930517439646721 threshold= 0.4467022009981762
训练 XGBoost ...


/vepfs-mlp2/xts001/400107/miniconda3/envs/ft_alphagenome/lib/python3.13/site-packages/xgboost/core.py:553: UserWarning: [21:05:42] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost Valid AUROC= 0.8892775742052099 threshold= 0.456790566444397
训练 LightGBM ...


/vepfs-mlp2/xts001/400107/miniconda3/envs/ft_alphagenome/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Valid AUROC= 0.8863503471735933 threshold= 0.3729506111506362
训练 DeepMLP ...
DeepMLP Valid AUROC= 0.8913616798256364 threshold= 0.5036038756370544


,model,n,positive,negative,AUROC,AUPRC,Balanced_Accuracy,F1,Sensitivity,Specificity,TN,FP,FN,TP,threshold,fit_seconds
0,ExtraTrees,27040,13520,13520,0.893052,0.894143,0.825481,0.825436,0.825222,0.825740,11164,2356,2363,11157,0.446702,1.735956
1,DeepMLP,27040,13520,13520,0.891362,0.892317,0.822411,0.822346,0.822041,0.822781,11124,2396,2406,11114,0.503604,19.096643
2,XGBoost,27040,13520,13520,0.889278,0.892157,0.822892,0.821924,0.817456,0.828328,11199,2321,2468,11052,0.456791,1.742872
3,LightGBM,27040,13520,13520,0.886350,0.888809,0.819194,0.821576,0.832544,0.805843,10895,2625,2264,11256,0.372951,2.251791
4,LogisticRegression,27040,13520,13520,0.884280,0.884960,0.820044,0.821457,0.827959,0.812130,10980,2540,2326,11194,0.530504,0.381183
5,RandomForest,27040,13520,13520,0.852183,0.860268,0.802367,0.795765,0.770044,0.834689,11285,2235,3109,10411,0.517922,1.790303


In [10]:
# ======================== 在接触Test前写出不可变选择锁 ========================
primary_model_name = str(validation_metrics.iloc[0]['model'])
primary_threshold = float(validation_metrics.iloc[0]['threshold'])

selected_id_hash = hashlib.sha256(
    ('\n'.join(sorted(selected_run.sample_id.astype(str))) + '\n').encode('utf-8')
).hexdigest()
feature_schema_hash = hashlib.sha256(
    feature_schema_path.read_bytes()
).hexdigest()
selection_lock = {
    'status': 'locked_before_test',
    'run_tag': RUN_TAG,
    'fast_mode': FAST_MODE,
    'feature_recipe': FEATURE_RECIPE,
    'models_requested': MODELS_TO_RUN,
    'models_actually_run': active_models,
    'primary_model': primary_model_name,
    'primary_threshold_from_validation': primary_threshold,
    'selection_rule': 'highest Valid AUROC; ties by Valid AUPRC then Balanced Accuracy',
    'threshold_rule': 'Youden J / maximum validation balanced accuracy',
    'test_used_for_model_selection': False,
    'test_used_for_threshold_selection': False,
    'refit_on_train_plus_valid_before_test': False,
    'test_evaluation_plan': 'predict all Test once, then exhaust majority class in disjoint 1:1 rounds',
    'balanced_round_metrics_aggregation': ['unweighted_mean', 'sample_size_weighted_mean'],
    'tss_evaluation_bins': ['ALL', '0-3kb', '3-12kb', '12-35kb', '>35kb'],
    'tss_used_as_model_feature': False,
    'selected_sample_id_sha256': selected_id_hash,
    'feature_schema_sha256': feature_schema_hash,
    'validation_ranking': validation_metrics.to_dict(orient='records'),
}
lock_path = RUN_DIR / 'selection_lock_before_test.json'
lock_path.write_text(json.dumps(selection_lock, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('选择锁已写入：', lock_path)
print('Primary model:', primary_model_name, 'Valid threshold:', primary_threshold)

选择锁已写入： /vepfs-mlp2/xts001/400107/code/AG_classification/GTEx_self/results/GTEX_all/selection_lock_before_test.json
Primary model: ExtraTrees Valid threshold: 0.4467022009981762


## Test解封边界

上一个单元已经把主模型、特征schema、分类阈值、全量Test与平衡轮次/TSS分区协议写入`selection_lock_before_test.json`。从下面开始才构造Test特征；不得返回上方根据Test结果改变模型、参数、feature recipe或阈值。所有轮次复用同一个锁定模型和Valid阈值。

In [11]:
# ======================== 锁定后评估全量Test和穷尽式1:1平衡轮次 ========================
if not (RUN_DIR / 'selection_lock_before_test.json').exists():
    raise RuntimeError('缺少selection lock，禁止评估Test')

# Test主表保留全部样本。平衡轮次只是评估视图，不改变或删除全量预测。
test_df = (
    selected_run.loc[selected_run.rf_split.eq('test')].copy()
    .reset_index(drop=True)
)
X_test = builder.transform(test_df)
y_test = test_df.label.to_numpy(np.int64)
unseen_test_tissues = sorted(
    set(test_df.target_tissue_standardized.astype(str)) - set(builder.tissue_categories)
) if FEATURE_RECIPE == 'score33_plus_tissue' else []
unknown_tissue_all_zero = True
if FEATURE_RECIPE == 'score33_plus_tissue' and unseen_test_tissues:
    unknown_mask = test_df.target_tissue_standardized.astype(str).isin(unseen_test_tissues).to_numpy()
    tissue_width = len(builder.tissue_categories)
    tissue_block = X_test[:, -tissue_width:] if tissue_width else np.empty((len(X_test), 0))
    unknown_tissue_all_zero = bool(np.all(tissue_block[unknown_mask] == 0))
    if not unknown_tissue_all_zero:
        raise AssertionError('Test未知Tissue没有编码为全0')
print('Test unseen tissues:', unseen_test_tissues)
print('Unknown-tissue one-hot all zero:', unknown_tissue_all_zero)

primary_model = fitted_models[primary_model_name]
test_probability = predict_probability(primary_model_name, primary_model, X_test)
test_prediction = (test_probability >= primary_threshold).astype(np.int8)

def cluster_bootstrap_auroc(frame, probability, n_bootstrap, seed):
    # 同一物理variant的多个context作为一个cluster一起重采样。
    work = frame[['variant_key', 'label']].copy().reset_index(drop=True)
    work['probability'] = probability
    grouped_indices = {
        key: group.index.to_numpy()
        for key, group in work.groupby('variant_key', sort=False)
    }
    keys = np.array(list(grouped_indices), dtype=object)
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_bootstrap):
        sampled_keys = rng.choice(keys, size=len(keys), replace=True)
        index = np.concatenate([grouped_indices[key] for key in sampled_keys])
        y = work.label.to_numpy()[index]
        p = work.probability.to_numpy()[index]
        if np.unique(y).size == 2:
            values.append(roc_auc_score(y, p))
    if len(values) < max(20, int(n_bootstrap * 0.8)):
        raise RuntimeError(f'有效bootstrap次数不足: {len(values)}/{n_bootstrap}')
    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975)), len(values)

def stable_order_for_test_round(frame, salt):
    key = frame.sample_id.astype(str).map(
        lambda value: hashlib.sha256(
            f'{RANDOM_SEED}|test_round|{salt}|{value}'.encode('utf-8')
        ).hexdigest()
    )
    return (
        frame.assign(_round_hash=key)
        .sort_values(['_round_hash', 'sample_id'], kind='stable')
        .drop(columns='_round_hash')
    )

def build_exhaustive_balanced_test_rounds(frame):
    # 多数类不放回地切块；少数类在完整轮次中复用，尾轮确定性抽取相同数量。
    positive = frame.loc[frame.label.eq(1)].copy()
    negative = frame.loc[frame.label.eq(0)].copy()
    if positive.empty or negative.empty:
        raise ValueError('Test必须同时包含正负样本')
    if len(negative) >= len(positive):
        majority_label, majority, minority = 0, negative, positive
    else:
        majority_label, majority, minority = 1, positive, negative
    majority = stable_order_for_test_round(majority, f'majority_label_{majority_label}')
    minority = stable_order_for_test_round(minority, f'minority_label_{1-majority_label}')

    rounds = []
    membership = []
    minority_size = len(minority)
    for round_number, start_row in enumerate(range(0, len(majority), minority_size), start=1):
        majority_chunk = majority.iloc[start_row:start_row + minority_size].copy()
        if len(majority_chunk) == minority_size:
            minority_chunk = minority.copy()
        else:
            minority_chunk = stable_order_for_test_round(
                minority, f'last_round_{round_number}'
            ).head(len(majority_chunk)).copy()
        round_frame = pd.concat([majority_chunk, minority_chunk], ignore_index=False)
        round_frame = round_frame.sort_values(
            ['chromosome', 'position', 'sample_id'], kind='stable'
        ).copy()
        round_id = f'test_round_{round_number:02d}'
        round_frame['test_round'] = round_id
        round_frame['_test_row_index'] = round_frame.index.astype(int)
        if round_frame.label.value_counts().nunique() != 1:
            raise AssertionError(f'{round_id}未达到严格1:1')
        rounds.append((round_id, round_frame))
        membership.append(round_frame[['_test_row_index', 'sample_id', 'label', 'test_round']])

    membership_table = pd.concat(membership, ignore_index=True)
    covered = set(membership_table.sample_id)
    if covered != set(frame.sample_id):
        raise AssertionError('平衡轮次没有覆盖全部Test sample_id')
    majority_ids = set(majority.sample_id)
    majority_membership = membership_table.loc[
        membership_table.label.eq(majority_label), 'sample_id'
    ]
    if len(majority_membership) != len(set(majority_membership)):
        raise AssertionError('多数类在不同Test轮次之间出现重复')
    if set(majority_membership) != majority_ids:
        raise AssertionError('多数类没有被全部且仅使用一次')
    return rounds, membership_table, majority_label

test_rounds, test_round_membership, test_majority_label = (
    build_exhaustive_balanced_test_rounds(test_df)
)
print(
    '全量Test:', len(test_df),
    'positive=', int((y_test == 1).sum()),
    'negative=', int((y_test == 0).sum()),
    'balanced_rounds=', len(test_rounds),
)

# 全量Test指标：AUROC不受类别比例影响；AUPRC保留真实Test prevalence。
test_result = metric_row(y_test, test_probability, primary_threshold)
n_bootstrap = FAST_BOOTSTRAP_REPLICATES if FAST_MODE else FULL_BOOTSTRAP_REPLICATES
ci_low, ci_high, n_valid_bootstrap = cluster_bootstrap_auroc(
    test_df, test_probability, n_bootstrap, RANDOM_SEED + 991
)
test_result.update({
    'evaluation_scope': 'all_test_full_imbalanced',
    'model': primary_model_name,
    'feature_recipe': FEATURE_RECIPE,
    'AUROC_CI95_low': ci_low,
    'AUROC_CI95_high': ci_high,
    'AUROC_CI_method': 'variant_key cluster bootstrap percentile',
    'bootstrap_requested': n_bootstrap,
    'bootstrap_valid': n_valid_bootstrap,
})
test_metrics = pd.DataFrame([test_result])
test_metrics.to_csv(RUN_DIR / 'test_metrics.csv', index=False)
test_metrics.to_csv(RUN_DIR / 'test_all_metrics.csv', index=False)

# 每个1:1轮次使用同一个Train模型和同一个Valid阈值，不重新训练或选阈值。
round_metric_rows = []
round_prediction_parts = []
for round_id, round_frame in test_rounds:
    row_index = round_frame._test_row_index.to_numpy(dtype=int)
    round_y = y_test[row_index]
    round_probability = test_probability[row_index]
    round_metric_rows.append({
        'test_round': round_id,
        **metric_row(round_y, round_probability, primary_threshold),
        'model': primary_model_name,
        'feature_recipe': FEATURE_RECIPE,
    })
    round_prediction = round_frame[[
        '_test_row_index', 'sample_id', 'variant_key', 'chromosome',
        'target_tissue_standardized', 'target_gene_id', 'gene_tss',
        'tss_distance', 'tss_mapping_method', 'label', 'test_round',
    ]].copy()
    round_prediction['probability_positive'] = round_probability
    round_prediction['predicted_label'] = (
        round_probability >= primary_threshold
    ).astype(np.int8)
    round_prediction_parts.append(round_prediction)

test_round_metrics = pd.DataFrame(round_metric_rows)
test_round_metrics.to_csv(RUN_DIR / 'test_round_metrics.csv', index=False)
round_predictions = pd.concat(round_prediction_parts, ignore_index=True)
round_predictions.to_parquet(
    RUN_DIR / 'test_round_predictions.parquet', index=False, compression='zstd'
)

averaged_metric_names = [
    'AUROC', 'AUPRC', 'Balanced_Accuracy', 'F1', 'Sensitivity', 'Specificity',
]
round_summary_rows = []
for metric_name in averaged_metric_names:
    values = pd.to_numeric(test_round_metrics[metric_name], errors='coerce')
    weights = test_round_metrics['n'].to_numpy(dtype=float)
    valid = values.notna().to_numpy()
    round_summary_rows.append({
        'metric': metric_name,
        'rounds': int(valid.sum()),
        'unweighted_mean': float(values[valid].mean()),
        'unweighted_std': float(values[valid].std(ddof=1)) if valid.sum() > 1 else 0.0,
        'sample_size_weighted_mean': float(
            np.average(values.to_numpy()[valid], weights=weights[valid])
        ),
    })
test_round_metrics_summary = pd.DataFrame(round_summary_rows)
test_round_metrics_summary.to_csv(
    RUN_DIR / 'test_round_metrics_summary.csv', index=False
)

# 全量Test逐样本预测只保存一次；轮次归属另存，避免把复用的少数类误当成独立样本。
prediction_table = test_df[[
    'sample_id', 'variant_key', 'chromosome', 'position',
    'target_tissue_standardized', 'target_gene_id', 'gene_tss',
    'tss_distance', 'tss_mapping_method', 'label', 'rf_split',
]].copy()
prediction_table['probability_positive'] = test_probability
prediction_table['predicted_label'] = test_prediction
prediction_table['locked_threshold'] = primary_threshold
prediction_table['primary_model'] = primary_model_name
prediction_table['feature_recipe'] = FEATURE_RECIPE
prediction_table.to_parquet(
    RUN_DIR / 'test_predictions.parquet', index=False, compression='zstd'
)
test_round_membership.to_parquet(
    RUN_DIR / 'test_round_assignments.parquet', index=False, compression='zstd'
)

pd.DataFrame(
    [[test_result['TN'], test_result['FP']], [test_result['FN'], test_result['TP']]],
    index=['true_negative', 'true_positive'], columns=['pred_negative', 'pred_positive'],
).to_csv(RUN_DIR / 'test_confusion_matrix.csv')

# TSS分区：ALL包含TSS缺失行；四个距离区间仅包含精确定位目标gene TSS的行。
TSS_EVALUATION_BINS = [
    ('ALL', None, None),
    ('0-3kb', 0, 3_000),
    ('3-12kb', 3_000, 12_000),
    ('12-35kb', 12_000, 35_000),
    ('>35kb', 35_000, None),
]

def calculate_tss_bin_metrics(frame, probability, evaluation_scope):
    distance = pd.to_numeric(frame.tss_distance, errors='coerce').abs()
    labels = frame.label.to_numpy(np.int64)
    rows = []
    for bin_order, (bin_name, lower, upper) in enumerate(TSS_EVALUATION_BINS):
        if bin_name == 'ALL':
            mask = np.ones(len(frame), dtype=bool)
        elif upper is None:
            mask = distance.ge(lower).fillna(False).to_numpy()
        else:
            mask = (
                distance.ge(lower).fillna(False).to_numpy()
                & distance.lt(upper).fillna(False).to_numpy()
            )
        y_bin = labels[mask]
        p_bin = np.asarray(probability)[mask]
        n_positive = int(np.sum(y_bin == 1))
        n_negative = int(np.sum(y_bin == 0))
        row = {
            'evaluation_scope': evaluation_scope,
            'model': primary_model_name,
            'tss_bin_order': bin_order,
            'tss_bin': bin_name,
            'n': int(mask.sum()),
            'positive': n_positive,
            'negative': n_negative,
            'tss_missing': int((mask & distance.isna().to_numpy()).sum()),
        }
        if (
            n_positive >= TSS_AUROC_MIN_POSITIVE
            and n_negative >= TSS_AUROC_MIN_NEGATIVE
        ):
            row.update(metric_row(y_bin, p_bin, primary_threshold))
        else:
            row.update({
                'AUROC': np.nan, 'AUPRC': np.nan,
                'Balanced_Accuracy': np.nan, 'F1': np.nan,
                'Sensitivity': np.nan, 'Specificity': np.nan,
                'TN': np.nan, 'FP': np.nan, 'FN': np.nan, 'TP': np.nan,
                'threshold': primary_threshold,
            })
        rows.append(row)
    return pd.DataFrame(rows)

tss_metric_parts = [
    calculate_tss_bin_metrics(test_df, test_probability, 'all_test_full_imbalanced')
]
for round_id, round_frame in test_rounds:
    row_index = round_frame._test_row_index.to_numpy(dtype=int)
    tss_metric_parts.append(
        calculate_tss_bin_metrics(
            round_frame, test_probability[row_index], round_id
        )
    )
test_tss_bin_metrics = pd.concat(tss_metric_parts, ignore_index=True)
test_tss_bin_metrics.to_csv(
    RUN_DIR / 'test_tss_bin_metrics.csv', index=False
)

round_tss = test_tss_bin_metrics.loc[
    test_tss_bin_metrics.evaluation_scope.str.startswith('test_round_')
].copy()
tss_summary_rows = []
for (bin_order, bin_name), group in round_tss.groupby(
    ['tss_bin_order', 'tss_bin'], sort=True
):
    summary_row = {
        'tss_bin_order': int(bin_order),
        'tss_bin': bin_name,
        'rounds': int(group.evaluation_scope.nunique()),
        'n_mean': float(group.n.mean()),
        'positive_mean': float(group.positive.mean()),
        'negative_mean': float(group.negative.mean()),
    }
    for metric_name in averaged_metric_names:
        values = pd.to_numeric(group[metric_name], errors='coerce')
        valid = values.notna().to_numpy()
        weights = group.n.to_numpy(dtype=float)
        summary_row[f'{metric_name}_mean'] = (
            float(values[valid].mean()) if valid.any() else np.nan
        )
        summary_row[f'{metric_name}_std'] = (
            float(values[valid].std(ddof=1)) if valid.sum() > 1 else 0.0
        )
        summary_row[f'{metric_name}_weighted_mean'] = (
            float(np.average(values.to_numpy()[valid], weights=weights[valid]))
            if valid.any() else np.nan
        )
    tss_summary_rows.append(summary_row)
test_tss_bin_metrics_summary = pd.DataFrame(tss_summary_rows).sort_values(
    'tss_bin_order'
)
test_tss_bin_metrics_summary.to_csv(
    RUN_DIR / 'test_tss_bin_metrics_summary.csv', index=False
)

tss_test_audit = (
    test_df.assign(tss_mapped=test_df.gene_tss.notna())
    .groupby(['label', 'sample_source', 'tss_mapping_method'], dropna=False)
    .agg(contexts=('sample_id', 'size'), tss_mapped=('tss_mapped', 'sum'))
    .reset_index()
)
tss_test_audit['coverage_pct'] = (
    100.0 * tss_test_audit.tss_mapped / tss_test_audit.contexts
)
tss_test_audit.to_csv(RUN_DIR / 'test_tss_coverage.csv', index=False)

display(test_metrics)
display(test_round_metrics)
display(test_round_metrics_summary)
display(test_tss_bin_metrics)

Test unseen tissues: []
Unknown-tissue one-hot all zero: True
全量Test: 33336 positive= 15551 negative= 17785 balanced_rounds= 2


,n,positive,negative,AUROC,AUPRC,Balanced_Accuracy,F1,Sensitivity,Specificity,TN,...,TP,threshold,evaluation_scope,model,feature_recipe,AUROC_CI95_low,AUROC_CI95_high,AUROC_CI_method,bootstrap_requested,bootstrap_valid
0,33336,15551,17785,0.890938,0.875767,0.823482,0.813546,0.826442,0.820523,14593,...,12852,0.446702,all_test_full_imbalanced,ExtraTrees,score33_plus_tissue,0.883691,0.897943,variant_key cluster bootstrap percentile,2000,2000


,test_round,n,positive,negative,AUROC,AUPRC,Balanced_Accuracy,F1,Sensitivity,Specificity,TN,FP,FN,TP,threshold,model,feature_recipe
0,test_round_01,31102,15551,15551,0.890928,0.888243,0.823323,0.823873,0.826442,0.820204,12755,2796,2699,12852,0.446702,ExtraTrees,score33_plus_tissue
1,test_round_02,4468,2234,2234,0.890538,0.886357,0.824082,0.824318,0.825425,0.822739,1838,396,390,1844,0.446702,ExtraTrees,score33_plus_tissue


,metric,rounds,unweighted_mean,unweighted_std,sample_size_weighted_mean
0,AUROC,2,0.890733,0.000276,0.890879
1,AUPRC,2,0.887300,0.001333,0.888006
2,Balanced_Accuracy,2,0.823703,0.000537,0.823419
3,F1,2,0.824095,0.000315,0.823929
4,Sensitivity,2,0.825934,0.000719,0.826314
5,Specificity,2,0.821472,0.001793,0.820523


,evaluation_scope,model,tss_bin_order,tss_bin,n,positive,negative,tss_missing,AUROC,AUPRC,Balanced_Accuracy,F1,Sensitivity,Specificity,TN,FP,FN,TP,threshold
0,all_test_full_imbalanced,ExtraTrees,0,ALL,33336,15551,17785,12816,0.890938,0.875767,0.823482,0.813546,0.826442,0.820523,14593,3192,2699,12852,0.446702
1,all_test_full_imbalanced,ExtraTrees,1,0-3kb,5241,4837,404,0,0.962603,0.996765,0.916737,0.934504,0.880504,0.952970,385,19,578,4259,0.446702
2,all_test_full_imbalanced,ExtraTrees,2,3-12kb,4436,3378,1058,0,0.949354,0.985468,0.897321,0.891684,0.807874,0.986767,1044,14,649,2729,0.446702
3,all_test_full_imbalanced,ExtraTrees,3,12-35kb,4686,3173,1513,0,0.944314,0.977533,0.890522,0.881724,0.792940,0.988103,1495,18,657,2516,0.446702
4,all_test_full_imbalanced,ExtraTrees,4,>35kb,6157,4016,2141,0,0.941646,0.974819,0.895868,0.887266,0.801544,0.990191,2120,21,797,3219,0.446702
5,test_round_01,ExtraTrees,0,ALL,31102,15551,15551,11267,0.890928,0.888243,0.823323,0.823873,0.826442,0.820204,12755,2796,2699,12852,0.446702
6,test_round_01,ExtraTrees,1,0-3kb,5185,4837,348,0,0.962080,0.997141,0.915827,0.934709,0.880504,0.951149,331,17,578,4259,0.446702
7,test_round_01,ExtraTrees,2,3-12kb,4285,3378,907,0,0.949431,0.987325,0.896771,0.891830,0.807874,0.985667,894,13,649,2729,0.446702
8,test_round_01,ExtraTrees,3,12-35kb,4492,3173,1319,0,0.944780,0.980006,0.890405,0.882033,0.792940,0.987870,1303,16,657,2516,0.446702
9,test_round_01,ExtraTrees,4,>35kb,5873,4016,1857,0,0.941911,0.977573,0.895656,0.887510,0.801544,0.989768,1838,19,797,3219,0.446702


In [12]:
# ======================== 保存全量、平衡轮次和TSS分区图 ========================
# 1) 全量Test ROC/PR：每条Test样本只出现一次。
fpr, tpr, _ = roc_curve(y_test, test_probability)
precision, recall, _ = precision_recall_curve(y_test, test_probability)

fig, ax = plt.subplots(figsize=(6.2, 5.6), constrained_layout=True)
ax.plot(fpr, tpr, linewidth=2.2, label=f'{primary_model_name} AUROC={test_result["AUROC"]:.4f}')
ax.fill_between([], [], [], label=f'95% CI {ci_low:.4f}–{ci_high:.4f}', alpha=0)
ax.plot([0, 1], [0, 1], '--', color='0.55', linewidth=1)
ax.set(
    xlabel='False Positive Rate', ylabel='True Positive Rate',
    title=f'GTEx full held-out Test ROC\n{FEATURE_RECIPE}',
)
ax.legend(frameon=False, loc='lower right')
fig.savefig(RUN_DIR / 'test_roc.png', dpi=220)
plt.close(fig)

prevalence = float(np.mean(y_test))
fig, ax = plt.subplots(figsize=(6.2, 5.6), constrained_layout=True)
ax.plot(recall, precision, linewidth=2.2, label=f'{primary_model_name} AUPRC={test_result["AUPRC"]:.4f}')
ax.axhline(prevalence, linestyle='--', color='0.55', linewidth=1, label=f'Prevalence={prevalence:.3f}')
ax.set(
    xlabel='Recall', ylabel='Precision',
    title=f'GTEx full held-out Test PR\n{FEATURE_RECIPE}',
)
ax.legend(frameon=False, loc='lower left')
fig.savefig(RUN_DIR / 'test_pr.png', dpi=220)
plt.close(fig)

# 2) 严格1:1平衡轮次：浅色为各轮，深色为逐轮曲线均值。
roc_grid = np.linspace(0.0, 1.0, 501)
roc_interpolated = []
fig, ax = plt.subplots(figsize=(6.5, 5.8), constrained_layout=True)
for round_id, round_frame in test_rounds:
    row_index = round_frame._test_row_index.to_numpy(dtype=int)
    round_y = y_test[row_index]
    round_probability = test_probability[row_index]
    round_fpr, round_tpr, _ = roc_curve(round_y, round_probability)
    roc_interpolated.append(np.interp(roc_grid, round_fpr, round_tpr))
    round_auc = roc_auc_score(round_y, round_probability)
    ax.plot(round_fpr, round_tpr, alpha=0.28, linewidth=1.2, label=f'{round_id} ({round_auc:.3f})')
mean_tpr = np.mean(roc_interpolated, axis=0)
mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
mean_round_auc = float(test_round_metrics.AUROC.mean())
ax.plot(roc_grid, mean_tpr, linewidth=2.8, color='C0', label=f'round mean (AUROC={mean_round_auc:.4f})')
ax.plot([0, 1], [0, 1], '--', color='0.55', linewidth=1)
ax.set(
    xlabel='False Positive Rate', ylabel='True Positive Rate',
    title='GTEx exhaustive balanced Test rounds ROC',
)
ax.legend(frameon=False, fontsize=8, loc='lower right')
fig.savefig(RUN_DIR / 'test_balanced_rounds_roc.png', dpi=220)
plt.close(fig)

recall_grid = np.linspace(0.0, 1.0, 501)
precision_interpolated = []
fig, ax = plt.subplots(figsize=(6.5, 5.8), constrained_layout=True)
for round_id, round_frame in test_rounds:
    row_index = round_frame._test_row_index.to_numpy(dtype=int)
    round_y = y_test[row_index]
    round_probability = test_probability[row_index]
    round_precision, round_recall, _ = precision_recall_curve(
        round_y, round_probability
    )
    precision_interpolated.append(
        np.interp(recall_grid, round_recall[::-1], round_precision[::-1])
    )
    round_auprc = average_precision_score(round_y, round_probability)
    ax.plot(round_recall, round_precision, alpha=0.28, linewidth=1.2, label=f'{round_id} ({round_auprc:.3f})')
mean_precision = np.mean(precision_interpolated, axis=0)
mean_round_auprc = float(test_round_metrics.AUPRC.mean())
ax.plot(
    recall_grid, mean_precision, linewidth=2.8, color='C1',
    label=f'round mean (AUPRC={mean_round_auprc:.4f})',
)
ax.axhline(0.5, linestyle='--', color='0.55', linewidth=1, label='Balanced prevalence=0.5')
ax.set(
    xlabel='Recall', ylabel='Precision',
    title='GTEx exhaustive balanced Test rounds PR',
)
ax.legend(frameon=False, fontsize=8, loc='lower left')
fig.savefig(RUN_DIR / 'test_balanced_rounds_pr.png', dpi=220)
plt.close(fig)

# 3) ALL和四个目标gene TSS距离区间。虚线为全量Test，实线为平衡轮次均值。
all_tss_rows = (
    test_tss_bin_metrics.loc[
        test_tss_bin_metrics.evaluation_scope.eq('all_test_full_imbalanced')
    ].sort_values('tss_bin_order')
)
tss_plot = test_tss_bin_metrics_summary.sort_values('tss_bin_order')
x = np.arange(len(tss_plot))
labels = tss_plot.tss_bin.astype(str).tolist()

for metric_name, output_name, ylabel in [
    ('AUROC', 'test_tss_bin_auroc.png', 'AUROC'),
    ('AUPRC', 'test_tss_bin_auprc.png', 'AUPRC'),
]:
    fig, ax = plt.subplots(figsize=(8.4, 5.8), constrained_layout=True)
    ax.plot(
        x, all_tss_rows[metric_name].to_numpy(dtype=float),
        marker='o', linestyle='--', linewidth=1.8,
        label='full Test',
    )
    ax.errorbar(
        x,
        tss_plot[f'{metric_name}_mean'].to_numpy(dtype=float),
        yerr=tss_plot[f'{metric_name}_std'].fillna(0).to_numpy(dtype=float),
        marker='o', linewidth=2.2, capsize=4,
        label='balanced rounds mean ± SD',
    )
    if metric_name == 'AUROC':
        ax.axhline(0.5, linestyle=':', color='0.55', linewidth=1)
    ax.set_xticks(x, labels)
    ax.set(
        xlabel='Absolute distance to target-gene TSS',
        ylabel=ylabel,
        title=f'GTEx Test {ylabel} by target-gene TSS distance',
        ylim=(0, 1),
    )
    ax.grid(axis='y', alpha=0.22)
    ax.legend(frameon=False)
    fig.savefig(RUN_DIR / output_name, dpi=220)
    plt.close(fig)

print(
    '已保存全量Test、平衡轮次和TSS分区ROC/PR图；',
    'TSS bins =', labels,
)

已保存全量Test、平衡轮次和TSS分区ROC/PR图； TSS bins = ['ALL', '0-3kb', '3-12kb', '12-35kb', '>35kb']


In [13]:
# ======================== 结果完整性与实验总摘要 ========================
required_outputs = [
    'validation_metrics.csv', 'selection_lock_before_test.json', 'feature_schema.json',
    'test_metrics.csv', 'test_all_metrics.csv', 'test_predictions.parquet',
    'test_round_metrics.csv', 'test_round_metrics_summary.csv',
    'test_round_predictions.parquet', 'test_round_assignments.parquet',
    'test_tss_bin_metrics.csv', 'test_tss_bin_metrics_summary.csv',
    'test_tss_coverage.csv', 'test_roc.png', 'test_pr.png',
    'test_balanced_rounds_roc.png', 'test_balanced_rounds_pr.png',
    'test_tss_bin_auroc.png', 'test_tss_bin_auprc.png',
    'selected_samples.parquet', 'chromosome_balance.csv',
]
missing_outputs = [name for name in required_outputs if not (RUN_DIR / name).exists()]
if missing_outputs:
    raise FileNotFoundError({'missing_outputs': missing_outputs})
if not any(MODELS_DIR.iterdir()):
    raise FileNotFoundError('models目录为空')

round_mean_lookup = {
    row.metric: {
        'unweighted_mean': float(row.unweighted_mean),
        'unweighted_std': float(row.unweighted_std),
        'sample_size_weighted_mean': float(row.sample_size_weighted_mean),
    }
    for row in test_round_metrics_summary.itertuples(index=False)
}
round_sizes = [
    {
        'test_round': str(row.test_round),
        'positive': int(row.positive),
        'negative': int(row.negative),
        'total': int(row.n),
    }
    for row in test_round_metrics.itertuples(index=False)
]
experiment_summary = {
    'status': 'smoke_test_complete' if FAST_MODE else 'full_training_complete',
    'run_tag': RUN_TAG,
    'fast_mode': FAST_MODE,
    'formal_six_model_training_started': bool(not FAST_MODE),
    'classification_unit': 'variant-target gene-target tissue/context',
    'feature_recipe': FEATURE_RECIPE,
    'models_requested': MODELS_TO_RUN,
    'models_actually_run': active_models,
    'primary_model': primary_model_name,
    'locked_threshold': primary_threshold,
    'train_rows': int(len(train_df)),
    'valid_rows': int(len(valid_df)),
    'test_rows_full': int(len(test_df)),
    'test_positive_full': int(test_df.label.eq(1).sum()),
    'test_negative_full': int(test_df.label.eq(0).sum()),
    'test_unique_variants': int(test_df.variant_key.nunique()),
    'test_all_samples_predicted_once': True,
    'test_balanced_rounds': {
        'count': int(len(test_rounds)),
        'round_sizes': round_sizes,
        'majority_label': int(test_majority_label),
        'majority_samples_disjoint_across_rounds': True,
        'all_test_sample_ids_covered_at_least_once': True,
        'minority_samples_may_repeat_across_rounds': bool(len(test_rounds) > 1),
        'metrics_summary': round_mean_lookup,
    },
    'tss_evaluation': {
        'definition': 'absolute hg38 distance from variant to exact target-gene TSS',
        'bins': ['ALL', '0-3kb', '3-12kb', '12-35kb', '>35kb'],
        'interval_rule': '[0,3kb), [3kb,12kb), [12kb,35kb), [35kb,infinity)',
        'all_includes_missing_tss': True,
        'distance_bins_exclude_missing_tss': True,
        'tss_mapped_rows': int(test_df.gene_tss.notna().sum()),
        'tss_coverage': float(test_df.gene_tss.notna().mean()),
        'used_as_model_feature': False,
    },
    'unseen_test_tissues': unseen_test_tissues,
    'unknown_tissue_one_hot_all_zero': unknown_tissue_all_zero,
    'test_metrics_full': test_result,
    'data_files': {
        'positive': str(POSITIVE_FILE),
        'negative': str(NEGATIVE_FILE),
        'selected_source': str(SELECTED_SAMPLES_FILE),
        'selected_actual_run': str(RUN_DIR / 'selected_samples.parquet'),
    },
    'strict_test_protocol': {
        'train_only_fit': True,
        'valid_only_model_selection': True,
        'valid_only_threshold_selection': True,
        'test_model_selection': False,
        'test_threshold_selection': False,
        'train_valid_refit': False,
        'same_locked_model_and_threshold_for_all_test_rounds': True,
    },
}
(RUN_DIR / 'experiment_summary.json').write_text(
    json.dumps(experiment_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(experiment_summary, ensure_ascii=False, indent=2))
if FAST_MODE:
    print('\n这是小规模RF流程测试；正式六模型训练没有启动。')

{
  "status": "full_training_complete",
  "run_tag": "GTEX_all",
  "fast_mode": false,
  "formal_six_model_training_started": true,
  "classification_unit": "variant-target gene-target tissue/context",
  "feature_recipe": "score33_plus_tissue",
  "models_requested": [
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees",
    "XGBoost",
    "LightGBM",
    "DeepMLP"
  ],
  "models_actually_run": [
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees",
    "XGBoost",
    "LightGBM",
    "DeepMLP"
  ],
  "primary_model": "ExtraTrees",
  "locked_threshold": 0.4467022009981762,
  "train_rows": 32519,
  "valid_rows": 27040,
  "test_rows_full": 33336,
  "test_positive_full": 15551,
  "test_negative_full": 17785,
  "test_unique_variants": 21583,
  "test_all_samples_predicted_once": true,
  "test_balanced_rounds": {
    "count": 2,
    "round_sizes": [
      {
        "test_round": "test_round_01",
        "positive": 15551,
        "negative": 15551,
        "total": 31102
 